# SensorGuard CUDA benchmark — n=15, with a permutation test

**Self-contained.** Upload this file to Colab and run it; nothing needs to be pushed to
GitHub first. It clones the repo for the unchanged pipeline, then overwrites three files
with the current local versions.

**Before you start: Runtime → Change runtime type → T4 GPU → Save.**

It does not touch the official test split. It fits on train and compares on validation only.

At the end, cell 10 prints a block to paste back into the chat.

In [ ]:
!nvidia-smi
import os
assert os.path.exists('/dev/nvidia0'), 'Select a Colab T4 GPU runtime, then Runtime > Run all.'

## 1. Clone and install the unchanged pipeline

In [ ]:
!rm -rf /content/sensorguard-ml
!git clone --branch agent/add-evidence-verifier --single-branch https://github.com/mghadia1/sensorguard-ml.git /content/sensorguard-ml
%cd /content/sensorguard-ml
%pip install -q -e .
import xgboost as xgb
print('XGBoost', xgb.__version__)
print('USE_CUDA:', xgb.build_info().get('USE_CUDA'))

## 2. Overwrite the three changed files

Editable install, so the next import picks these up.

In [ ]:
%%writefile src/sensorguard/gpu_benchmark.py
"""Reproducible CPU-versus-CUDA XGBoost benchmark for a Colab GPU runtime."""

from __future__ import annotations

import itertools
import json
import math
import platform
import statistics
import subprocess
import time
from pathlib import Path
from typing import Any

import numpy as np
import xgboost as xgb
from sklearn.metrics import average_precision_score, roc_auc_score
from xgboost import XGBClassifier

from .data import DatasetSplits, feature_target
from .modeling import make_preprocessor, select_threshold, xgboost_parameters

#: Threshold frozen by the August 2026 XGBoost comparison. Agreement between the
#: CPU and CUDA models only matters relative to a decision boundary: two
#: probabilities either side of this produce opposite predictions.
FROZEN_DECISION_THRESHOLD = 0.66

#: Enumerate every split below this many; above it, sample.
EXACT_PERMUTATION_LIMIT = 200_000
MONTE_CARLO_PERMUTATIONS = 100_000

#: Guards float equality when counting permutations at least as extreme.
_TIE_TOLERANCE = 1e-12


def permutation_p_value(
    cpu: list[float],
    cuda: list[float],
    *,
    alternative: str = "cuda_faster",
    random_state: int = 42,
) -> tuple[float, bool, int]:
    """Exact one-sided permutation test on the median difference when the split
    count is tractable, otherwise a seeded Monte Carlo approximation.

    Returns (p_value, exact: bool, n_permutations: int).

    Under the null hypothesis the device label is arbitrary, so every way of
    relabelling the pooled timings is equally likely. The p-value is the share of
    relabellings whose median difference is at least as extreme as the observed
    one. This assumes nothing about normality, which matters because fit times on
    a shared Colab CPU are strongly right-skewed.
    """
    if alternative != "cuda_faster":
        raise ValueError(f"unsupported alternative: {alternative!r}")
    if len(cpu) < 1 or len(cuda) < 1:
        raise ValueError("both device timing lists must be non-empty")

    pooled = np.asarray(list(cpu) + list(cuda), dtype=float)
    n_cpu, n_total = len(cpu), len(pooled)
    observed = float(np.median(cpu)) - float(np.median(cuda))

    total_splits = math.comb(n_total, n_cpu)
    exact = total_splits <= EXACT_PERMUTATION_LIMIT

    if exact:
        indices = np.arange(n_total)
        rows = []
        for combination in itertools.combinations(range(n_total), n_cpu):
            mask = np.zeros(n_total, dtype=bool)
            mask[list(combination)] = True
            rows.append(np.concatenate([indices[mask], indices[~mask]]))
        index_matrix = np.asarray(rows)
        n_permutations = total_splits
    else:
        rng = np.random.default_rng(random_state)
        index_matrix = np.argsort(
            rng.random((MONTE_CARLO_PERMUTATIONS, n_total)), axis=1
        )
        n_permutations = MONTE_CARLO_PERMUTATIONS

    permuted = pooled[index_matrix]
    differences = np.median(permuted[:, :n_cpu], axis=1) - np.median(
        permuted[:, n_cpu:], axis=1
    )
    at_least_as_extreme = int(np.count_nonzero(differences >= observed - _TIE_TOLERANCE))

    if exact:
        p_value = at_least_as_extreme / n_permutations
    else:
        # (count + 1) / (n + 1) keeps a sampled p-value away from an
        # unsupportable exact zero; the floor is 1 / (n + 1).
        p_value = (at_least_as_extreme + 1) / (n_permutations + 1)
    return float(p_value), exact, int(n_permutations)


def _spread_ratio(runs: list[float]) -> float:
    """max / min — how far the slowest repeat sat from the fastest."""
    return float(max(runs) / min(runs))


def _stdev(runs: list[float]) -> float:
    return float(statistics.stdev(runs)) if len(runs) > 1 else 0.0


def _timed_fit(
    features: Any,
    labels: Any,
    *,
    device: str,
    random_state: int,
    scale_pos_weight: float,
    repeats: int,
) -> tuple[XGBClassifier, list[float], float]:
    """Fit ``repeats`` times after one discarded warm-up.

    The warm-up exists because the first CUDA fit pays for context creation. In
    the August 2026 five-repeat run the first CUDA fit took 0.731 s against about
    0.417 s for the other four — that is setup cost, not compute, and leaving it
    in the sample drags the median.
    """

    def _fit_once() -> tuple[XGBClassifier, float]:
        model = XGBClassifier(
            **xgboost_parameters(
                device=device,
                random_state=random_state,
                scale_pos_weight=scale_pos_weight,
            )
        )
        started = time.perf_counter()
        model.fit(features, labels)
        return model, time.perf_counter() - started

    _, warmup_seconds = _fit_once()

    durations: list[float] = []
    model: XGBClassifier | None = None
    for _ in range(repeats):
        model, elapsed = _fit_once()
        durations.append(elapsed)
    assert model is not None
    return model, durations, warmup_seconds


def _nvidia_smi() -> str:
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
            check=True,
            capture_output=True,
            text=True,
            timeout=15,
        )
    except (FileNotFoundError, subprocess.CalledProcessError, subprocess.TimeoutExpired) as error:
        raise RuntimeError(
            "No usable NVIDIA runtime was detected. In Colab choose Runtime > "
            "Change runtime type > T4 GPU, then run all cells again."
        ) from error
    value = result.stdout.strip()
    if not value:
        raise RuntimeError("nvidia-smi returned no GPU information")
    return value


def disagreement_at_threshold(
    cpu_probabilities: Any, cuda_probabilities: Any, threshold: float
) -> dict[str, Any]:
    """Rows where the two devices land on opposite sides of the decision boundary.

    A maximum absolute probability difference says nothing on its own: a 0.28 gap
    is harmless at 0.05 versus 0.33 and decisive at 0.55 versus 0.83. This counts
    the rows where the two models would actually disagree.
    """
    cpu_predictions = np.asarray(cpu_probabilities) >= threshold
    cuda_predictions = np.asarray(cuda_probabilities) >= threshold
    disagreeing = int(np.count_nonzero(cpu_predictions != cuda_predictions))
    total = int(len(cpu_predictions))
    return {
        "threshold": float(threshold),
        "rows": total,
        "disagreeing_rows": disagreeing,
        "disagreeing_fraction": (disagreeing / total) if total else 0.0,
    }


def run_gpu_benchmark(
    splits: DatasetSplits,
    output_path: str | Path,
    *,
    random_state: int = 42,
    repeats: int = 15,
) -> dict[str, Any]:
    """Compare frozen CPU/CUDA XGBoost settings without touching the test split."""

    if repeats < 1:
        raise ValueError("repeats must be at least one")
    gpu = _nvidia_smi()
    train_features, train_labels = feature_target(splits.train)
    validation_features, validation_labels = feature_target(splits.validation)
    positive_count = int(np.asarray(train_labels).sum())
    if positive_count < 1:
        raise ValueError("training split must contain at least one positive example")
    scale_pos_weight = (len(train_labels) - positive_count) / positive_count

    preprocessing_started = time.perf_counter()
    preprocessor = make_preprocessor()
    transformed_train = preprocessor.fit_transform(train_features)
    transformed_validation = preprocessor.transform(validation_features)
    preprocessing_seconds = time.perf_counter() - preprocessing_started

    cpu_model, cpu_fit_seconds, cpu_warmup = _timed_fit(
        transformed_train,
        train_labels,
        device="cpu",
        random_state=random_state,
        scale_pos_weight=scale_pos_weight,
        repeats=repeats,
    )
    try:
        cuda_model, cuda_fit_seconds, cuda_warmup = _timed_fit(
            transformed_train,
            train_labels,
            device="cuda",
            random_state=random_state,
            scale_pos_weight=scale_pos_weight,
            repeats=repeats,
        )
    except xgb.core.XGBoostError as error:
        raise RuntimeError(
            "XGBoost could not train on CUDA. Confirm that Colab is using a GPU "
            "runtime and that the full xgboost package, not xgboost-cpu, is installed."
        ) from error

    cpu_probabilities = cpu_model.predict_proba(transformed_validation)[:, 1]
    cuda_probabilities = cuda_model.predict_proba(transformed_validation)[:, 1]
    cpu_median = float(np.median(cpu_fit_seconds))
    cuda_median = float(np.median(cuda_fit_seconds))
    p_value, exact, n_permutations = permutation_p_value(
        cpu_fit_seconds, cuda_fit_seconds, random_state=random_state
    )

    # Same validation-only protocol as the CPU sweep, run independently on each
    # device's probabilities. If the two disagree, that is the finding.
    cpu_selected_threshold = select_threshold(validation_labels, cpu_probabilities)
    cuda_selected_threshold = select_threshold(validation_labels, cuda_probabilities)

    agreement = {
        "cpu_average_precision": float(
            average_precision_score(validation_labels, cpu_probabilities)
        ),
        "cuda_average_precision": float(
            average_precision_score(validation_labels, cuda_probabilities)
        ),
        "cpu_roc_auc": float(roc_auc_score(validation_labels, cpu_probabilities)),
        "cuda_roc_auc": float(roc_auc_score(validation_labels, cuda_probabilities)),
        "maximum_absolute_probability_difference": float(
            np.max(np.abs(cpu_probabilities - cuda_probabilities))
        ),
        "disagreement_at_threshold": disagreement_at_threshold(
            cpu_probabilities, cuda_probabilities, FROZEN_DECISION_THRESHOLD
        ),
        "cpu_selected_threshold": float(cpu_selected_threshold),
        "cuda_selected_threshold": float(cuda_selected_threshold),
        "note": (
            "These are two different models, not one model on two devices. "
            "XGBoost's CPU and CUDA hist implementations sketch quantiles "
            "differently, so agreement is measured rather than assumed."
        ),
    }

    report: dict[str, Any] = {
        "status": "verified_cuda_run",
        "protocol": {
            "purpose": "CPU versus CUDA implementation agreement and timing",
            "dataset_partition": "train for fitting; validation for agreement metrics",
            "official_test_evaluated": False,
            "preprocessing_fit_on": "training split only",
            "random_state": random_state,
            "repeats": repeats,
            "warmup_fits_discarded_per_device": 1,
            "frozen_decision_threshold": FROZEN_DECISION_THRESHOLD,
            "xgboost": xgboost_parameters(
                device="cuda",
                random_state=random_state,
                scale_pos_weight=scale_pos_weight,
            ),
        },
        "environment": {
            "gpu": gpu,
            "python": platform.python_version(),
            "platform": platform.platform(),
            "xgboost": xgb.__version__,
            "xgboost_build_info": xgb.build_info(),
        },
        "rows": {
            "train": int(len(train_labels)),
            "validation": int(len(validation_labels)),
            "test_evaluated": 0,
        },
        "timing_seconds": {
            "shared_preprocessing": preprocessing_seconds,
            "cpu_fit_runs": cpu_fit_seconds,
            "cuda_fit_runs": cuda_fit_seconds,
            "cpu_fit_median": cpu_median,
            "cuda_fit_median": cuda_median,
            "cpu_over_cuda_speedup": cpu_median / cuda_median,
            "cpu_fit_stdev": _stdev(cpu_fit_seconds),
            "cuda_fit_stdev": _stdev(cuda_fit_seconds),
            "cpu_fit_spread_ratio": _spread_ratio(cpu_fit_seconds),
            "cuda_fit_spread_ratio": _spread_ratio(cuda_fit_seconds),
            "speedup_p_value": p_value,
            "speedup_test_exact": exact,
            "speedup_test_permutations": n_permutations,
            "warmup_seconds": {"cpu": float(cpu_warmup), "cuda": float(cuda_warmup)},
        },
        "validation_agreement": agreement,
        # Deprecated alias, retained for one release so evidence files written
        # against the old key still verify.
        "validation_parity": agreement,
        "interpretation": (
            "A speedup above 1.0 means CUDA was faster by median fit time. Read it "
            "beside speedup_p_value: on this small tabular dataset the two timing "
            "distributions overlap heavily, and a median ratio alone does not "
            "establish a difference."
        ),
    }
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    return report

In [ ]:
%%writefile src/sensorguard/evidence.py
"""Integrity checks for the published SensorGuard CUDA benchmark evidence."""

from __future__ import annotations

import json
import math
import re
import statistics
from pathlib import Path
from typing import Any

from .gpu_benchmark import permutation_p_value

#: A benchmark with fewer repeats than this cannot separate a noisy shared CPU
#: from a GPU. The August 2026 five-repeat run reached p = 0.0595 against a
#: floor of 0.0238 for that design — not a power ceiling, just too few repeats.
MINIMUM_REPEATS_PER_DEVICE = 10


SHA256_PATTERN = re.compile(r"^[0-9a-f]{64}$")


def _finite_positive(value: Any, label: str) -> float:
    number = float(value)
    if not math.isfinite(number) or number <= 0.0:
        raise ValueError(f"{label} must be finite and positive")
    return number


def _unit_interval(value: Any, label: str) -> float:
    number = float(value)
    if not math.isfinite(number) or not 0.0 <= number <= 1.0:
        raise ValueError(f"{label} must be between zero and one")
    return number


def verify_cuda_evidence(path: str | Path) -> dict[str, Any]:
    """Validate provenance, protocol guards, timing arithmetic, and metrics."""

    evidence_path = Path(path)
    payload = json.loads(evidence_path.read_text(encoding="utf-8"))
    if payload.get("status") != "verified_cuda_run":
        raise ValueError("evidence status is not a verified CUDA run")
    source_hash = payload.get("source_file_sha256", "")
    if not SHA256_PATTERN.fullmatch(str(source_hash)):
        raise ValueError("source report SHA-256 is missing or malformed")

    protocol = payload["protocol"]
    rows = payload["rows"]
    environment = payload["environment"]
    if protocol.get("official_test_evaluated") is not False:
        raise ValueError("CUDA comparison must not evaluate the official test split")
    if int(rows.get("test_evaluated", -1)) != 0:
        raise ValueError("CUDA comparison reports official-test access")
    if int(rows.get("train", 0)) != 6000 or int(rows.get("validation", 0)) != 2000:
        raise ValueError("benchmark split sizes differ from the frozen protocol")
    if protocol["xgboost"].get("device") != "cuda":
        raise ValueError("published XGBoost device is not CUDA")
    if environment["xgboost_build_info"].get("USE_CUDA") is not True:
        raise ValueError("XGBoost build does not report CUDA support")

    repeats = int(protocol["repeats"])
    timing = payload["timing_seconds"]
    cpu_runs = [_finite_positive(value, "CPU fit time") for value in timing["cpu_fit_runs"]]
    cuda_runs = [
        _finite_positive(value, "CUDA fit time") for value in timing["cuda_fit_runs"]
    ]
    if repeats < 1 or len(cpu_runs) != repeats or len(cuda_runs) != repeats:
        raise ValueError("timing-run counts differ from the declared repeat count")
    if len(cpu_runs) != len(cuda_runs):
        raise ValueError(
            f"device run counts differ: {len(cpu_runs)} CPU vs {len(cuda_runs)} CUDA"
        )
    if len(cpu_runs) < MINIMUM_REPEATS_PER_DEVICE:
        raise ValueError(
            f"underpowered benchmark: {len(cpu_runs)} timed runs per device, "
            f"minimum is {MINIMUM_REPEATS_PER_DEVICE}"
        )

    cpu_median = float(statistics.median(cpu_runs))
    cuda_median = float(statistics.median(cuda_runs))
    if not math.isclose(cpu_median, float(timing["cpu_fit_median"]), rel_tol=1e-12):
        raise ValueError("published CPU median does not match the raw runs")
    if not math.isclose(cuda_median, float(timing["cuda_fit_median"]), rel_tol=1e-12):
        raise ValueError("published CUDA median does not match the raw runs")
    speedup = cpu_median / cuda_median
    if not math.isclose(speedup, float(timing["cpu_over_cuda_speedup"]), rel_tol=1e-12):
        raise ValueError("published CUDA speedup does not match the raw timings")

    # Recomputed, not trusted. A verifier that only rechecks the numbers it was
    # handed is a checksum; recomputing the p-value is what makes it evidence.
    for label, runs, published in (
        ("CPU", cpu_runs, timing["cpu_fit_stdev"]),
        ("CUDA", cuda_runs, timing["cuda_fit_stdev"]),
    ):
        if not math.isclose(statistics.stdev(runs), float(published), rel_tol=1e-9):
            raise ValueError(f"published {label} stdev does not match the raw runs")
    for label, runs, published in (
        ("CPU", cpu_runs, timing["cpu_fit_spread_ratio"]),
        ("CUDA", cuda_runs, timing["cuda_fit_spread_ratio"]),
    ):
        if not math.isclose(max(runs) / min(runs), float(published), rel_tol=1e-9):
            raise ValueError(f"published {label} spread ratio does not match the raw runs")

    recomputed_p, exact, permutations = permutation_p_value(
        cpu_runs, cuda_runs, random_state=int(protocol["random_state"])
    )
    if not math.isclose(recomputed_p, float(timing["speedup_p_value"]), rel_tol=1e-9):
        raise ValueError("published speedup p-value does not match the raw runs")
    if bool(timing["speedup_test_exact"]) is not exact:
        raise ValueError("published permutation test exactness does not match the raw runs")
    if int(timing["speedup_test_permutations"]) != permutations:
        raise ValueError("published permutation count does not match the raw runs")

    warmup = timing["warmup_seconds"]
    for device, runs in (("cpu", cpu_runs), ("cuda", cuda_runs)):
        value = _finite_positive(warmup[device], f"{device} warm-up time")
        if any(math.isclose(value, run, rel_tol=1e-12) for run in runs):
            raise ValueError(
                f"{device} warm-up fit appears in the timed runs; it must be discarded"
            )

    agreement = payload.get("validation_agreement", payload.get("validation_parity"))
    if agreement is None:
        raise ValueError("evidence has no validation_agreement block")
    for key in (
        "cpu_average_precision",
        "cuda_average_precision",
        "cpu_roc_auc",
        "cuda_roc_auc",
        "maximum_absolute_probability_difference",
    ):
        _unit_interval(agreement[key], key)

    disagreement = agreement.get("disagreement_at_threshold")
    if disagreement is None:
        raise ValueError("evidence does not report disagreement_at_threshold")
    disagreeing = int(disagreement["disagreeing_rows"])
    total_rows = int(disagreement["rows"])
    if total_rows != int(rows["validation"]):
        raise ValueError("threshold disagreement row count differs from the validation split")
    if not 0 <= disagreeing <= total_rows:
        raise ValueError("threshold disagreement count is out of range")
    if not math.isclose(
        disagreeing / total_rows, float(disagreement["disagreeing_fraction"]), rel_tol=1e-9
    ):
        raise ValueError("published disagreement fraction does not match the counts")

    return {
        "status": "verified",
        "gpu": environment["gpu"],
        "xgboost": environment["xgboost"],
        "repeats": repeats,
        "cpu_fit_median_seconds": cpu_median,
        "cuda_fit_median_seconds": cuda_median,
        "cpu_over_cuda_speedup": speedup,
        "speedup_p_value": recomputed_p,
        "speedup_test_exact": exact,
        "speedup_test_permutations": permutations,
        "cpu_fit_spread_ratio": max(cpu_runs) / min(cpu_runs),
        "cuda_fit_spread_ratio": max(cuda_runs) / min(cuda_runs),
        "disagreeing_rows_at_frozen_threshold": disagreeing,
        "official_test_rows_evaluated": 0,
        "source_file_sha256": source_hash,
    }

In [ ]:
%%writefile src/sensorguard/cli.py
"""Command-line interface for data, training, comparison, and inference."""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import pandas as pd

from .data import load_dataset, split_dataset, validate_dataset
from .download import download_dataset
from .evidence import verify_cuda_evidence
from .gpu_benchmark import run_gpu_benchmark
from .learning import run_interactive_check
from .modeling import load_bundle, predict_rows, train_evaluate_save
from .torch_model import train_torch_comparison


def main() -> int:
    parser = argparse.ArgumentParser(description="SensorGuard predictive-maintenance ML pipeline")
    subparsers = parser.add_subparsers(dest="command", required=True)

    download_parser = subparsers.add_parser("download", help="download and verify the UCI dataset")
    download_parser.add_argument("--destination", type=Path, default=Path("data/raw"))
    download_parser.add_argument("--force", action="store_true")

    audit_parser = subparsers.add_parser("audit", help="validate and summarize the dataset")
    audit_parser.add_argument("--data", type=Path, default=Path("data/raw/ai4i2020.csv"))

    train_parser = subparsers.add_parser("train", help="train, select, and evaluate classical models")
    train_parser.add_argument("--data", type=Path, default=Path("data/raw/ai4i2020.csv"))
    train_parser.add_argument("--out", type=Path, default=Path("outputs/baseline"))
    train_parser.add_argument("--random-state", type=int, default=42)
    train_parser.add_argument("--with-torch", action="store_true")
    train_parser.add_argument("--torch-epochs", type=int, default=40)

    predict_parser = subparsers.add_parser("predict", help="run batch predictions from a CSV file")
    predict_parser.add_argument("--model", type=Path, required=True)
    predict_parser.add_argument("--input", type=Path, required=True)
    predict_parser.add_argument("--out", type=Path, required=True)

    learn_parser = subparsers.add_parser("learn", help="complete the interactive ML-readiness check")
    learn_parser.add_argument(
        "--out",
        type=Path,
        default=Path("outputs/learning-check/answers.json"),
    )

    gpu_parser = subparsers.add_parser(
        "gpu-benchmark", help="compare frozen CPU and CUDA XGBoost on train/validation"
    )
    gpu_parser.add_argument("--data", type=Path, default=Path("data/raw/ai4i2020.csv"))
    gpu_parser.add_argument(
        "--out", type=Path, default=Path("outputs/cuda-benchmark/report.json")
    )
    gpu_parser.add_argument("--random-state", type=int, default=42)
    gpu_parser.add_argument("--repeats", type=int, default=15)

    evidence_parser = subparsers.add_parser(
        "verify-evidence", help="audit the published CUDA benchmark evidence"
    )
    evidence_parser.add_argument(
        "--report",
        type=Path,
        default=Path("docs/evidence/cuda-colab-t4-report.json"),
    )

    args = parser.parse_args()
    if args.command == "download":
        path = download_dataset(args.destination, force=args.force)
        print(f"Dataset ready: {path}")
        return 0
    if args.command == "audit":
        frame = load_dataset(args.data)
        print(json.dumps(validate_dataset(frame), indent=2))
        return 0
    if args.command == "train":
        frame = load_dataset(args.data)
        audit = validate_dataset(frame)
        splits = split_dataset(frame, random_state=args.random_state)
        report = train_evaluate_save(splits, args.out, random_state=args.random_state)
        report["dataset_audit"] = audit
        if args.with_torch:
            report["torch_comparison"] = train_torch_comparison(
                splits,
                args.out,
                random_state=args.random_state,
                epochs=args.torch_epochs,
            )
        args.out.mkdir(parents=True, exist_ok=True)
        (args.out / "experiment_summary.json").write_text(
            json.dumps(report, indent=2) + "\n",
            encoding="utf-8",
        )
        test = report["test_metrics"]
        print(
            f"Selected {report['selected_model']} at threshold {report['selected_threshold']:.2f}; "
            f"test precision={test['precision']:.3f} recall={test['recall']:.3f} "
            f"F1={test['f1']:.3f} AP={test['average_precision']:.3f}"
        )
        return 0
    if args.command == "predict":
        bundle = load_bundle(args.model)
        rows = pd.read_csv(args.input)
        results = predict_rows(bundle, rows)
        args.out.parent.mkdir(parents=True, exist_ok=True)
        results.to_csv(args.out, index=False)
        print(f"Wrote {len(results)} predictions to {args.out}")
        return 0
    if args.command == "learn":
        run_interactive_check(args.out)
        return 0
    if args.command == "gpu-benchmark":
        frame = load_dataset(args.data)
        splits = split_dataset(frame, random_state=args.random_state)
        report = run_gpu_benchmark(
            splits,
            args.out,
            random_state=args.random_state,
            repeats=args.repeats,
        )
        timing = report["timing_seconds"]
        print(
            f"Verified CUDA run; median CPU fit={timing['cpu_fit_median']:.4f}s, "
            f"CUDA fit={timing['cuda_fit_median']:.4f}s, "
            f"speedup={timing['cpu_over_cuda_speedup']:.3f}x. "
            f"Wrote {args.out}"
        )
        return 0
    if args.command == "verify-evidence":
        # A failed audit is a result, not a crash: report it plainly and exit
        # non-zero so CI and a human read the same sentence.
        try:
            result = verify_cuda_evidence(args.report)
        except (ValueError, KeyError) as error:
            print(f"Evidence rejected: {error}")
            return 1
        print(json.dumps(result, indent=2))
        return 0
    raise AssertionError(f"unsupported command: {args.command}")


if __name__ == "__main__":
    raise SystemExit(main())

## 3. Prove the right code is loaded — before spending GPU time

The known-answer test pins the permutation implementation to `p = 0.0595` on the August 5-repeat data. If this fails, stop: the numbers below would be meaningless.

In [ ]:
%%writefile tests/test_cuda_statistics.py
"""Standalone statistics checks. Run these BEFORE spending GPU time."""

import statistics
import unittest

import sensorguard.gpu_benchmark as gb
from sensorguard.gpu_benchmark import disagreement_at_threshold, permutation_p_value

AUGUST_CPU_RUNS = [0.368, 0.806, 0.855, 1.105, 1.580]
AUGUST_CUDA_RUNS = [0.413, 0.416, 0.418, 0.419, 0.731]


class PermutationTests(unittest.TestCase):
    def test_reproduces_published_august_p_value(self):
        """Known answer: the underpowered 5v5 run gives exactly 15/252."""
        p, exact, n = permutation_p_value(AUGUST_CPU_RUNS, AUGUST_CUDA_RUNS)
        self.assertTrue(exact)
        self.assertEqual(n, 252)
        self.assertAlmostEqual(p, 0.0595, places=4)

    def test_identical_inputs_give_no_evidence(self):
        p, _, _ = permutation_p_value([1.0, 2, 3, 4, 5], [1.0, 2, 3, 4, 5])
        self.assertGreater(p, 0.5)

    def test_separated_inputs_hit_the_attainable_floor(self):
        p, _, _ = permutation_p_value([10.0, 11, 12, 13, 14], [1.0, 2, 3, 4, 5])
        self.assertAlmostEqual(p, 6 / 252, places=6)

    def test_exact_and_monte_carlo_agree(self):
        cpu, cuda = [0.9, 1.0, 1.1, 1.2, 1.3], [0.4, 0.5, 0.6, 0.7, 0.8]
        exact_p, _, _ = permutation_p_value(cpu, cuda)
        original = gb.EXACT_PERMUTATION_LIMIT
        gb.EXACT_PERMUTATION_LIMIT = 1
        try:
            sampled_p, is_exact, _ = permutation_p_value(cpu, cuda)
        finally:
            gb.EXACT_PERMUTATION_LIMIT = original
        self.assertFalse(is_exact)
        self.assertLess(abs(exact_p - sampled_p), 0.01)


class ThresholdTests(unittest.TestCase):
    def test_identical_probabilities_never_disagree(self):
        p = [0.1, 0.5, 0.66, 0.9]
        self.assertEqual(disagreement_at_threshold(p, p, 0.66)["disagreeing_rows"], 0)

    def test_counts_rows_that_straddle_the_boundary(self):
        result = disagreement_at_threshold([0.55, 0.10, 0.80], [0.83, 0.38, 0.90], 0.66)
        self.assertEqual(result["disagreeing_rows"], 1)


if __name__ == "__main__":
    unittest.main(verbosity=2)

In [ ]:
!python -m pytest -q tests/test_cuda_statistics.py

## 4. Data

In [ ]:
!sensorguard download --destination data/raw
!sensorguard audit --data data/raw/ai4i2020.csv

## 5. The benchmark — 15 timed repeats per device, 1 discarded warm-up each

Takes a few minutes.

In [ ]:
!sensorguard gpu-benchmark --data data/raw/ai4i2020.csv --out outputs/cuda-benchmark/report.json --random-state 42 --repeats 15

## 6. Independent audit

Recomputes the medians, stdevs, spread ratios and the p-value from the raw runs. Must print `"status": "verified"`.

In [ ]:
!sensorguard verify-evidence --report outputs/cuda-benchmark/report.json

## 7. The numbers

In [ ]:
import json
from pathlib import Path

report = json.loads(Path("outputs/cuda-benchmark/report.json").read_text())
t = report["timing_seconds"]
a = report["validation_agreement"]
d = a["disagreement_at_threshold"]

print("=" * 66)
print("PASTE THIS BLOCK BACK INTO THE CHAT")
print("=" * 66)
print(f"gpu                    : {report['environment']['gpu']}")
print(f"xgboost                : {report['environment']['xgboost']}")
print(f"timed repeats / device : {report['protocol']['repeats']}")
print(f"warmup discarded (s)   : cpu={t['warmup_seconds']['cpu']:.4f} cuda={t['warmup_seconds']['cuda']:.4f}")
print("-" * 66)
print(f"cpu median fit (s)     : {t['cpu_fit_median']:.4f}")
print(f"cuda median fit (s)    : {t['cuda_fit_median']:.4f}")
print(f"X  speedup             : {t['cpu_over_cuda_speedup']:.4f}x")
print(f"P  p-value             : {t['speedup_p_value']:.4f}  (exact={t['speedup_test_exact']}, n={t['speedup_test_permutations']})")
print(f"A  cpu spread ratio    : {t['cpu_fit_spread_ratio']:.2f}x   (stdev {t['cpu_fit_stdev']:.4f})")
print(f"B  cuda spread ratio   : {t['cuda_fit_spread_ratio']:.2f}x   (stdev {t['cuda_fit_stdev']:.4f})")
print("-" * 66)
print(f"K  disagreeing rows    : {d['disagreeing_rows']}")
print(f"M  validation rows     : {d['rows']}   ({d['disagreeing_fraction']:.4%})")
print(f"D  max |prob diff|     : {a['maximum_absolute_probability_difference']:.4f}")
print(f"cpu selected threshold : {a['cpu_selected_threshold']:.2f}")
print(f"cuda selected threshold: {a['cuda_selected_threshold']:.2f}")
print(f"official test rows     : {report['rows']['test_evaluated']}")
print("=" * 66)
print()
print("VERDICT:", "significant at 0.05" if t["speedup_p_value"] < 0.05
      else f"NOT established at n={report['protocol']['repeats']} (p = {t['speedup_p_value']:.4f})")


## 8. Download the report

Save it into the repo at `projects/sensorguard-ml/docs/evidence/cuda-colab-t4-report-n15.json`, then say "done" in the chat.

In [ ]:
from google.colab import files
files.download('outputs/cuda-benchmark/report.json')